In [1]:
from google.colab import drive
import os
import pandas as pd
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Copying to Colabs local SSD
!cp -r "/content/drive/MyDrive/MMRetrieval/flickr8k" /content/

In [3]:
#DATASET_DIR = "/content/drive/MyDrive/MMRetrieval/flickr8k"
IMAGE_DIR = "/content/flickr8k/Images"
CAPTION_FILE = "/content/flickr8k/captions.txt"

In [4]:
print("Images:", len(os.listdir(IMAGE_DIR)))

df = pd.read_csv(CAPTION_FILE)

print(df.head())
print(df.columns)
print(df.shape)

Images: 8091
                       image  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  
Index(['image', 'caption'], dtype='object')
(40455, 2)


In [5]:
images = df["image"].unique().tolist()

print("Unique images:", len(images))

Unique images: 8091


In [6]:
# Create fixed split, 80, 10, 10
train_imgs, temp_imgs = train_test_split(
    images,
    train_size=6000,
    random_state=42,
    shuffle=True
)

val_imgs, test_imgs = train_test_split(
    temp_imgs,
    test_size=1000,
    random_state=42,
    shuffle=True
)

print(len(train_imgs))
print(len(val_imgs))
print(len(test_imgs))

6000
1091
1000


In [7]:
# Train test and val dataset
train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)

val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)

test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

In [8]:
# Save files in drive
import pickle

split_dict = {
    "train": train_imgs,
    "val": val_imgs,
    "test": test_imgs
}

with open(
    "/content/flickr8k/flickr8k_split.pkl",
    "wb"
) as f:
    pickle.dump(split_dict, f)

In [9]:
# Test Load
with open(
    "/content/flickr8k/flickr8k_split.pkl",
    "rb"
) as f:
    split_dict = pickle.load(f)

In [10]:
train_df.head()

,image,caption
0,1001773457_577c3a7d70.jpg,A black dog and a spotted dog are fighting
1,1001773457_577c3a7d70.jpg,A black dog and a tri-colored dog playing with...
2,1001773457_577c3a7d70.jpg,A black dog and a white dog with brown spots a...
3,1001773457_577c3a7d70.jpg,Two dogs of different breeds looking at each o...
4,1001773457_577c3a7d70.jpg,Two dogs on pavement moving toward each other .


Data Preprocessing

In [11]:
import re
from collections import Counter
import pickle


In [12]:
# Clean captions

def clean_caption(text):
    text = text.lower()

    # remove punctuation
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [13]:
# Apply on the datasets

train_df["caption"] = train_df["caption"].apply(clean_caption)
val_df["caption"] = val_df["caption"].apply(clean_caption)
test_df["caption"] = test_df["caption"].apply(clean_caption)

In [14]:
# Add the specialised tokens

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"

In [15]:
# Train data vocab

counter = Counter()

for caption in train_df["caption"]:
    counter.update(caption.split())

In [16]:
print("Unique words:", len(counter))

Unique words: 7744


In [17]:
# Create Vocab

MIN_FREQ = 5

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
    SOS_TOKEN: 2,
    EOS_TOKEN: 3
}

for word, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

idx2word = {idx: word for word, idx in vocab.items()}

In [18]:
print("Vocabulary size:", len(vocab))

Vocabulary size: 2561


In [19]:
# Encoding

def encode_caption(text, vocab):

    tokens = text.split()

    encoded = [vocab["<SOS>"]]

    for token in tokens:
        encoded.append(
            vocab.get(token, vocab["<UNK>"])
        )

    encoded.append(vocab["<EOS>"])

    return encoded

In [20]:
# Set Max caption length for batching

lengths = []

for caption in train_df["caption"]:
    lengths.append(
        len(caption.split()) + 2
    )

print("Max length:", max(lengths))
print("Average length:", sum(lengths)/len(lengths))

Max length: 38
Average length: 12.790533333333334


In [21]:
MAX_LEN = 38

In [22]:
# Save and reload vocab

with open(
    "/content/flickr8k/vocab.pkl",
    "wb"
) as f:
    pickle.dump(vocab, f)

In [23]:
with open(
    "/content/flickr8k/vocab.pkl",
    "rb"
) as f:
    vocab = pickle.load(f)

Dataset Classes

In [24]:
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image
import os
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

In [25]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [26]:
from collections import defaultdict
import random

train_caption_map = defaultdict(list)
val_caption_map = defaultdict(list)
test_caption_map = defaultdict(list)

bad_img = "861608773_bdafd5c996.jpg"

train_caption_map.pop(bad_img, None)
val_caption_map.pop(bad_img, None)
test_caption_map.pop(bad_img, None)

for _, row in train_df.iterrows():
    train_caption_map[row["image"]].append(row["caption"])

for _, row in val_df.iterrows():
    val_caption_map[row["image"]].append(row["caption"])

for _, row in test_df.iterrows():
    test_caption_map[row["image"]].append(row["caption"])


class Flickr8KRetrievalDataset(Dataset):
    def __init__(self, caption_map, image_dir, vocab, transform=None, random_caption=True):
        self.image_names = sorted(caption_map.keys())
        self.caption_map = caption_map
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform
        self.random_caption = random_caption

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]

        captions = self.caption_map[image_name]

        if self.random_caption:
            caption = random.choice(captions)
        else:
            caption = captions[0]      # fixed caption for eval

        try:
          image = Image.open(
              os.path.join(self.image_dir, image_name)
          ).convert("RGB")

        except Exception:
            return self.__getitem__(
                (idx + 1) % len(self)
            )

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, caption

In [27]:
class Flickr8KAllCaptionEvalDataset(Dataset):

    def __init__(self, caption_map, image_dir, vocab, transform=None):

        self.samples = []
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform

        for image_name in sorted(caption_map.keys()):

            for caption in caption_map[image_name]:

                self.samples.append(
                    (image_name, caption)
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        image_name, caption = self.samples[idx]

        image = Image.open(
            os.path.join(self.image_dir, image_name)
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, torch.tensor(caption)

In [93]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    images = []
    captions = []
    lengths = []

    for image, caption in batch:
        images.append(image)
        captions.append(caption)
        lengths.append(len(caption))

    images = torch.stack(images)

    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    lengths = torch.tensor(lengths)

    return images, captions, lengths

In [94]:
train_dataset = Flickr8KRetrievalDataset(
    train_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform,
    random_caption=True
)

val_dataset = Flickr8KRetrievalDataset(
    val_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform,
    random_caption=False
)

test_dataset = Flickr8KRetrievalDataset(
    test_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform,
    random_caption=False
)

In [95]:
all_caption_test_dataset = Flickr8KAllCaptionEvalDataset(
    test_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform
)

In [96]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

6000
1091
1000


In [97]:
# Dataloaders for clip style loading
BATCH_SIZE = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

In [98]:
all_caption_test_loader = DataLoader(
    all_caption_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

In [99]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [100]:
device

'cuda'

In [101]:
print("device" in globals())
print("text_encoder" in globals())
print("train_loader" in globals())

True
True
True


In [102]:
image, caption = train_dataset[0]

print(image.shape)

print(caption)

torch.Size([3, 224, 224])
tensor([ 2, 26, 27, 28, 29, 30, 31, 23, 14, 15, 16, 17, 18,  3])


ViT(vit_base_patch16_224) Image Encoder

In [103]:
EMBED_DIM = 256

In [104]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

class ViTEncoder(nn.Module):

    def __init__(self, freeze=True):
        super().__init__()
        self.freeze = freeze # Store freeze as an instance attribute

        # Original ViT-B/16 (Dosovitskiy et al., 2020)
        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0
        )

        # Freeze ViT initially
        if self.freeze:
            for param in self.vit.parameters():
                param.requires_grad = False

        # Projection Head
        self.projection = nn.Sequential(
            nn.Linear(768, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512)
        )

    def forward(self, images):

        # ViT Features
        if self.freeze:
            with torch.no_grad():
                features = self.vit(images)
        else:
            features = self.vit(images)

        # Projection
        embeddings = self.projection(features)   # (B,512)

        # Normalize
        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

Replace the entire LSTM encoder with a Transformer Encoder while keeping the output as a 512-dimensional embedding so that it remains compatible with existing ViT encoder and Bidirectional InfoNCE loss.

In [105]:
# Text Encoder (Transformer Encoder)

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float()
            * (-math.log(10000.0) / embed_dim)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)      # (1, max_len, embed_dim)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TextEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        num_heads=8,
        num_layers=4,
        ff_dim=2048,
        pad_idx=0,
        dropout=0.1
    ):
        super().__init__()

        # Word Embedding
        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        # Positional Encoding
        self.position = PositionalEncoding(embed_dim)

        # Transformer Encoder Layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        # Transformer Encoder
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Projection Head
        self.projection = nn.Sequential(
            nn.Linear(embed_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512)
        )

    def forward(
        self,
        captions,
        lengths
    ):

        # Word Embeddings
        x = self.embedding(captions)

        # Positional Encoding
        x = self.position(x)

        # Padding Mask
        device = captions.device

        max_len = captions.size(1)

        lengths = lengths.to(device)

        mask = (
            torch.arange(max_len, device=device)
            .expand(len(lengths), max_len)
            >= lengths.unsqueeze(1)
        )

        # Transformer Encoder
        x = self.transformer(
            x,
            src_key_padding_mask=mask
        )

        # Mean Pooling (ignore padding)
        mask_float = (~mask).unsqueeze(-1).float()

        x = (x * mask_float).sum(dim=1) / mask_float.sum(dim=1)

        # Projection
        embeddings = self.projection(x)

        # Normalize
        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [106]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


In [107]:
encoder = ViTEncoder().to(device)

images, captions, lengths = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    image_embeddings = encoder(images)

print(image_embeddings.shape)

torch.Size([256, 512])


In [108]:
# text encoder installation and test
text_encoder = TextEncoder(
    vocab_size=len(vocab),
    embed_dim=512,
    num_heads=8,
    num_layers=4,
    ff_dim=2048,
    pad_idx=vocab["<PAD>"],
    dropout=0.1
).to(device)

images, captions, lengths = next(iter(train_loader))

captions = captions.to(device)

with torch.no_grad():
    text_embeddings = text_encoder(
        captions,
        lengths
    )

print(text_embeddings.shape)

torch.Size([256, 512])


In [109]:
# compatibility verification
with torch.no_grad():
    image_embeddings = encoder(images.to(device))

    text_embeddings = text_encoder(
        captions.to(device),
        lengths
    )

print(image_embeddings.shape)
print(text_embeddings.shape)

torch.Size([256, 512])
torch.Size([256, 512])


In [110]:
# Normalize embedding
import torch.nn.functional as F

image_embeddings = F.normalize(image_embeddings, dim=1)
text_embeddings = F.normalize(text_embeddings, dim=1)

print(image_embeddings.norm(dim=1).mean())
print(text_embeddings.norm(dim=1).mean())

tensor(1., device='cuda:0')
tensor(1., device='cuda:0')


In [111]:
# Joint Model (ViT + Transformer Retrieval)

import torch
import torch.nn as nn
import numpy as np

class ViTTransformerRetrieval(nn.Module):

    def __init__(self, image_encoder, text_encoder):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # Learnable temperature parameter (CLIP-style)
        self.logit_scale = nn.Parameter(
            torch.ones([]) * np.log(1 / 0.07)
        )

    def forward(self, images, captions, lengths):

        # Image embeddings from ViT
        image_emb = self.image_encoder(images)

        # Caption embeddings from Transformer Encoder
        text_emb = self.text_encoder(captions, lengths)

        return image_emb, text_emb


# Build Model
model = ViTTransformerRetrieval(
    image_encoder=encoder,
    text_encoder=text_encoder
).to(device)

In [112]:
# CLIP-Style Contrastive Loss

def clip_contrastive_loss(image_emb, text_emb, logit_scale):

    # CLIP-style learnable temperature
    logit_scale = logit_scale.exp().clamp(max=100)

    # Similarity Matrix
    logits = torch.matmul(image_emb, text_emb.T) * logit_scale

    # Ground truth labels
    targets = torch.arange(
        image_emb.size(0),
        device=image_emb.device
    )

    # Image → Text
    loss_i2t = F.cross_entropy(logits, targets)

    # Text → Image
    loss_t2i = F.cross_entropy(logits.T, targets)

    # Bidirectional InfoNCE
    loss = (loss_i2t + loss_t2i) / 2

    return loss, logits

In [113]:
# Forward pass test
images, captions, lengths = next(iter(train_loader))

images = images.to(device)
captions = captions.to(device)

with torch.no_grad():
    image_emb, text_emb = model(
        images,
        captions,
        lengths
    )

print(image_emb.shape)
print(text_emb.shape)

torch.Size([256, 512])
torch.Size([256, 512])


In [114]:
# Loss test
loss, logits = clip_contrastive_loss(
    image_emb,
    text_emb,
    model.logit_scale
)

print(loss.item())
print(logits.shape)

5.680750846862793
torch.Size([256, 256])


In [115]:
# Training
optimizer = torch.optim.AdamW(
    [
        {
            "params": model.image_encoder.vit.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.image_encoder.projection.parameters(),
            "lr": 3e-4
        },
        {
            "params": model.text_encoder.parameters(),
            "lr": 3e-4
        },
        {
            "params": [model.logit_scale],
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50
)

In [116]:
# Add Validation Loop
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, captions, lengths in dataloader:
            images = images.to(device)
            captions = captions.to(device)

            image_emb, text_emb = model(
                images,
                captions,
                lengths
            )

            loss, _ = clip_contrastive_loss(
                image_emb,
                text_emb,
                model.logit_scale
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [117]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
del images, captions
gc.collect()
torch.cuda.empty_cache()

In [118]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-80GB


In [119]:
print(len(train_loader))

24


In [120]:
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 24
Validation batches: 5


In [121]:
print(train_dataset.image_dir)

/content/flickr8k/Images


In [122]:
import time
import torch

# Training
NUM_EPOCHS = 50
best_val_loss = float("inf")

BEST_MODEL_PATH = "/content/drive/MyDrive/MMRetrieval/ViT_Transformer_m1/best_retrieval_ViTTransformer_model.pth"

for epoch in range(NUM_EPOCHS):

    print(f"\n================ Epoch {epoch+1}/{NUM_EPOCHS} ================")

    model.train()
    running_loss = 0.0

    epoch_start = time.time()

    for batch_idx, (images, captions, lengths) in enumerate(train_loader):

        batch_start = time.time()

        images = images.to(device, non_blocking=True)
        captions = captions.to(device, non_blocking=True)

        optimizer.zero_grad()

        image_emb, text_emb = model(
            images,
            captions,
            lengths
        )

        loss, _ = clip_contrastive_loss(
            image_emb,
            text_emb,
            model.logit_scale
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()

        batch_time = time.time() - batch_start

        print(
            f"Batch {batch_idx+1:02d}/{len(train_loader)} | "
            f"Loss: {loss.item():.4f} | "
            f"Time: {batch_time:.2f}s"
        )

    train_time = time.time() - epoch_start

    train_loss = running_loss / len(train_loader)

    # ---------------- Validation ----------------
    val_start = time.time()

    val_loss = evaluate(
        model,
        val_loader,
        device
    )

    val_time = time.time() - val_start

    scheduler.step()

    print("\n---------------- Summary ----------------")
    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Training Time  : {train_time:.2f} sec")
    print(f"Validation Time: {val_time:.2f} sec")
    print(
        f"GPU Memory Used: "
        f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
    )

    # Save best model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss
            },
            BEST_MODEL_PATH
        )

        print("✓ Best model saved")


================ Epoch 1/50 ================
Batch 01/24 | Loss: 5.6817 | Time: 0.58s
Batch 02/24 | Loss: 5.5937 | Time: 0.56s
Batch 03/24 | Loss: 5.5767 | Time: 0.56s
Batch 04/24 | Loss: 5.5777 | Time: 0.55s
Batch 05/24 | Loss: 5.5438 | Time: 0.56s
Batch 06/24 | Loss: 5.5182 | Time: 0.56s
Batch 07/24 | Loss: 5.4804 | Time: 0.56s
Batch 08/24 | Loss: 5.3299 | Time: 0.56s
Batch 09/24 | Loss: 5.2639 | Time: 0.55s
Batch 10/24 | Loss: 5.1471 | Time: 0.56s
Batch 11/24 | Loss: 5.1916 | Time: 0.55s
Batch 12/24 | Loss: 5.1738 | Time: 0.56s
Batch 13/24 | Loss: 5.0440 | Time: 0.56s
Batch 14/24 | Loss: 4.9941 | Time: 0.55s
Batch 15/24 | Loss: 4.8726 | Time: 0.56s
Batch 16/24 | Loss: 4.8243 | Time: 0.56s
Batch 17/24 | Loss: 4.8547 | Time: 0.56s
Batch 18/24 | Loss: 4.8274 | Time: 0.55s
Batch 19/24 | Loss: 4.8069 | Time: 0.56s
Batch 20/24 | Loss: 4.8147 | Time: 0.56s
Batch 21/24 | Loss: 4.7495 | Time: 0.56s
Batch 22/24 | Loss: 4.7129 | Time: 0.56s
Batch 23/24 | Loss: 4.5732 | Time: 0.56s
Batch 24/24

In [123]:
FINAL_MODEL_PATH = "/content/drive/MyDrive/MMRetrieval/ViT_Transformer_m1/best_retrieval_ViTTransformer_model.pth"

torch.save(model.state_dict(), BEST_MODEL_PATH)

Evaluation

In [124]:
def extract_embeddings(model, dataloader, device):
    model.load_state_dict(
    torch.load(FINAL_MODEL_PATH, map_location=device)
    )

    model.eval()


    image_embeddings = []
    text_embeddings = []

    with torch.no_grad():
        for images, captions, lengths in dataloader:
            images = images.to(device)
            captions = captions.to(device)

            img_emb, txt_emb = model(images, captions, lengths)

            image_embeddings.append(img_emb.cpu())
            text_embeddings.append(txt_emb.cpu())

    image_embeddings = torch.cat(image_embeddings, dim=0)
    text_embeddings = torch.cat(text_embeddings, dim=0)

    return image_embeddings, text_embeddings

In [125]:
image_embs, text_embs = extract_embeddings(
    model,
    all_caption_test_loader,
    device
)

print(image_embs.shape)
print(text_embs.shape)

/tmp/ipykernel_10276/340369310.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return image, torch.tensor(caption)


torch.Size([5000, 512])
torch.Size([5000, 512])


In [126]:
# Similarity Matrix
unique_image_embs = image_embs[::5]

similarity = unique_image_embs @ text_embs.T

print("Image embeddings:", unique_image_embs.shape)
print("Text embeddings:", text_embs.shape)
print("Similarity:", similarity.shape)

Image embeddings: torch.Size([1000, 512])
Text embeddings: torch.Size([5000, 512])
Similarity: torch.Size([1000, 5000])


In [127]:
# image -> text
def image_to_text_recall(similarity, k):

    correct = 0

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        topk = similarity[img_idx].topk(k).indices.tolist()

        if any(idx in gt_caps for idx in topk):
            correct += 1

    return correct / similarity.shape[0]


In [128]:
def text_to_image_recall(similarity, k):

    similarity_t = similarity.T

    correct = 0

    for cap_idx in range(similarity_t.shape[0]):

        gt_img = cap_idx // 5

        topk = similarity_t[cap_idx]\
            .topk(k)\
            .indices\
            .tolist()

        if gt_img in topk:
            correct += 1

    return correct / similarity_t.shape[0]


In [129]:
def image_to_text_mrr(similarity):

    reciprocal_ranks = []

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        sorted_idx = torch.argsort(
            similarity[img_idx],
            descending=True
        )

        best_rank = float("inf")

        for cap in gt_caps:

            rank = (
                (sorted_idx == cap)
                .nonzero(as_tuple=True)[0]
                .item()
            ) + 1

            best_rank = min(best_rank, rank)

        reciprocal_ranks.append(
            1.0 / best_rank
        )

    return np.mean(reciprocal_ranks)

In [130]:
def text_to_image_mrr(similarity):

    similarity_t2i = similarity.T

    reciprocal_ranks = []

    for cap_idx in range(similarity_t2i.shape[0]):

        gt_image = cap_idx // 5

        sorted_idx = torch.argsort(
            similarity_t2i[cap_idx],
            descending=True
        )

        rank = (
            (sorted_idx == gt_image)
            .nonzero(as_tuple=True)[0]
            .item()
        ) + 1

        reciprocal_ranks.append(
            1.0 / rank
        )

    return np.mean(reciprocal_ranks)

In [131]:
results = pd.DataFrame({
    "Metric": [
        "Recall@1",
        "Recall@5",
        "Recall@10",
        "MRR"
    ],
    "Image→Text": [
        image_to_text_recall(similarity, 1),
        image_to_text_recall(similarity, 5),
        image_to_text_recall(similarity, 10),
        image_to_text_mrr(similarity)
    ],
    "Text→Image": [
        text_to_image_recall(similarity, 1),
        text_to_image_recall(similarity, 5),
        text_to_image_recall(similarity, 10),
        text_to_image_mrr(similarity)
    ]
})

results["Image→Text"] = results["Image→Text"].round(6)
results["Text→Image"] = results["Text→Image"].round(6)

display(results)

,Metric,Image→Text,Text→Image
0,Recall@1,0.297000,0.226600
1,Recall@5,0.589000,0.503000
2,Recall@10,0.704000,0.641400
3,MRR,0.431402,0.359915
